# Intron AfriSpeech-200 Automatic Speech Recognition Challenge
**Objective**: Create an automatic speech recognition (ASR) model for African accents.
**Metric**: Word Error Rate (WER)

### Important Notes for Kaggle:
1. **Storage**: The full training set is 50GB, which exceeds Kaggle's 20GB limit for `/kaggle/working`. This notebook is designed to work with a subset of the data or by processing files in-place if possible.
2. **Dependencies**: You need to install `jiwer` for the WER metric.

In [ ]:
!pip install jiwer datasets transformers torchaudio

In [ ]:
import os
import zipfile
import torch
import torchaudio
import pandas as pd
from datasets import Dataset, Audio, load_metric
from transformers import Wav2Vec2ForCTC, Wav2Vec2Processor, Trainer, TrainingArguments
import numpy as np
import re
from tqdm import tqdm

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

## 1. Path Configuration and Data Extraction

In [ ]:
def setup_paths():
    train_csv = 'train_metadata.csv'
    train_zip = 'afrispeech-train.zip'
    dev_zip = 'afrispeech-dev.zip'
    test_metadata = 'test_metadata.csv'
    
    if os.path.exists('/kaggle/input'):
        for root, dirs, files in os.walk('/kaggle/input'):
            if 'train_metadata.csv' in files: train_csv = os.path.join(root, 'train_metadata.csv')
            if 'afrispeech-train.zip' in files: train_zip = os.path.join(root, 'afrispeech-train.zip')
            if 'afrispeech-dev.zip' in files: dev_zip = os.path.join(root, 'afrispeech-dev.zip')
            if 'test_metadata.csv' in files: test_metadata = os.path.join(root, 'test_metadata.csv')
                
    return train_csv, train_zip, dev_zip, test_metadata

TRAIN_CSV, TRAIN_ZIP, DEV_ZIP, TEST_CSV = setup_paths()
AUDIO_DIR = '/kaggle/working/audio'

def extract_subset(zip_path, extract_path, num_files=1000):
    """Extract only a subset of files to stay within Kaggle storage limits"""
    if not os.path.exists(extract_path):
        os.makedirs(extract_path, exist_ok=True)
        print(f"Extracting subset from {zip_path}...")
        with zipfile.ZipFile(zip_path, 'r') as z:
            files = z.namelist()[:num_files]
            for f in tqdm(files):
                z.extract(f, extract_path)
        print("Extraction complete.")
    else:
        print("Audio already exists.")

# For demonstration, we extract a small part of the dev/train set
if os.path.exists(DEV_ZIP):
    extract_subset(DEV_ZIP, AUDIO_DIR, num_files=500)

## 2. Data Preparation Utils

In [ ]:
def remove_special_characters(batch):
    batch["transcription"] = re.sub(r'[\\,\\.\\!\\?\\-\\;\\:\\"\\(\\)]', '', batch["transcription"]).lower() + " "
    return batch

def prepare_dataset(batch):
    audio = batch["audio"]
    batch["input_values"] = processor(audio["array"], sampling_rate=audio["sampling_rate"]).input_values[0]
    
    with processor.as_target_processor():
        batch["labels"] = processor(batch["transcription"]).input_ids
    return batch

## 3. Model and Trainer Setup

In [ ]:
MODEL_ID = "facebook/wav2vec2-large-xlsr-53"
processor = Wav2Vec2Processor.from_pretrained(MODEL_ID)
model = Wav2Vec2ForCTC.from_pretrained(
    MODEL_ID, 
    attention_dropout=0.1,
    hidden_dropout=0.1,
    feat_proj_dropout=0.0,
    mask_time_prob=0.05,
    layerdrop=0.1,
    gradient_checkpointing=True, 
    ctc_loss_reduction="mean", 
    pad_token_id=processor.tokenizer.pad_token_id,
    vocab_size=len(processor.tokenizer)
).to(device)

## 4. Evaluation Function

In [ ]:
wer_metric = load_metric("wer")

def compute_metrics(pred):
    pred_logits = pred.predictions
    pred_ids = np.argmax(pred_logits, axis=-1)
    pred.label_ids[pred.label_ids == -100] = processor.tokenizer.pad_token_id
    pred_str = processor.batch_decode(pred_ids)
    label_str = processor.batch_decode(pred.label_ids, group_tokens=False)
    wer = wer_metric.compute(predictions=pred_str, references=label_str)
    return {"wer": wer}

## 5. Main Training Execution

In [ ]:
if os.path.exists(TRAIN_CSV):
    df = pd.read_csv(TRAIN_CSV)
    # Filter rows where audio exists in our subset
    df['path'] = df['path'].apply(lambda x: os.path.join(AUDIO_DIR, os.path.basename(x)))
    df = df[df['path'].apply(os.path.exists)]
    
    if len(df) > 0:
        dataset = Dataset.from_pandas(df)
        dataset = dataset.map(remove_special_characters)
        dataset = dataset.cast_column("path", Audio(sampling_rate=16_000))
        dataset = dataset.rename_column("path", "audio")
        dataset = dataset.map(prepare_dataset, remove_columns=dataset.column_names)
        
        splits = dataset.train_test_split(test_size=0.1)
        
        training_args = TrainingArguments(
          output_dir="./wav2vec2-afrispeech",
          group_by_length=True,
          per_device_train_batch_size=4,
          gradient_accumulation_steps=4,
          num_train_epochs=5,
          fp16=True,
          save_steps=500,
          eval_steps=500,
          logging_steps=10,
          learning_rate=1e-4,
          warmup_steps=500,
          save_total_limit=2,
          push_to_hub=False,
          report_to="none"
        )
        
        from transformers import DataCollatorForCTCWithPadding
        data_collator = DataCollatorForCTCWithPadding(processor=processor, padding=True)
        
        trainer = Trainer(
            model=model,
            data_collator=data_collator,
            args=training_args,
            compute_metrics=compute_metrics,
            train_dataset=splits["train"],
            eval_dataset=splits["test"],
            tokenizer=processor.feature_extractor,
        )
        
        print("Starting training...")
        trainer.train()
    else:
        print("No matching audio files found for training.")
else:
    print("train_metadata.csv not found.")

## 6. Inference and Submission Generation

In [ ]:
if os.path.exists(TEST_CSV):
    test_df = pd.read_csv(TEST_CSV)
    # Assuming we have a way to access test audio files
    # test_df['path'] = ...
    
    results = []
    model.eval()
    
    # This is a placeholder for actual inference logic on the full test set
    print("Inference placeholder: In a real run, you would iterate over test files and transcribe.")
    
    # Example output format:
    # sub = pd.DataFrame({'ID': test_ids, 'Text': transcriptions})
    # sub.to_csv('submission.csv', index=False)
else:
    print("Test metadata not found.")